# Developmental LM — Phase 1 training

This notebook trains only the Phase 1 causal GRU next-phoneme model. Timing is fixed at 10 ms/frame, 5 frames/phoneme, and 1 frame overlap. Real training requires a North American English IPA-CHILDES export, a complete IPA feature table, and an empirical utterance-pause file.

## 1. Runtime setup
In Colab, select **Runtime → Change runtime type → T4 GPU** if desired. The baseline also runs on CPU, although the current minimal trainer does not yet include explicit device placement.

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = "https://github.com/ss-sebastian/developmental_checkpoints_word_recognition.git"
PROJECT = pathlib.Path("/content/developmental_checkpoints_word_recognition")
if PROJECT.exists():
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{PROJECT}[panphon]", "pytest"], check=True)
os.chdir(PROJECT)
print("Project ready:", PROJECT)

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], check=True)

## 2. Optional smoke training
This uses only the repository's synthetic test fixtures and verifies the training/checkpoint path. It is not scientific training data.

In [ ]:
RUN_SMOKE = True
if RUN_SMOKE:
    subprocess.run([sys.executable, "-m", "devlm.cli", "--config", "configs/smoke.toml"], check=True)

## 3. Upload real-data files to the Colab runtime
Upload the North American English IPA-CHILDES export, complete IPA feature table, and empirical utterance-pause JSON one at a time. No empirical phoneme-duration file is needed. Inputs and checkpoints stay under `/content`; they will be lost if the runtime disconnects before the final ZIP download.

In [ ]:
from google.colab import files

UPLOAD_DIR = pathlib.Path("/content/devlm_inputs")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

def upload_one(label):
    print(f"Select exactly one file for: {label}")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Expected exactly one file for {label}; rerun this cell")
    name, content = next(iter(uploaded.items()))
    destination = UPLOAD_DIR / pathlib.Path(name).name
    destination.write_bytes(content)
    print(f"Saved {label}: {destination}")
    return str(destination)

DATASET_PATH = upload_one("North American English IPA-CHILDES export")
FEATURE_TABLE_PATH = upload_one("complete IPA feature mapping JSON")
PAUSE_DURATIONS_PATH = upload_one("empirical utterance-pause JSON")
OUTPUT_DIR = "/content/devlm_phase1_outputs"
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## 4. Write the real-training configuration
The five-value envelope is symmetric and normalized by the implementation to peak 1.0. Phoneme extent and overlap remain fixed and are not configurable.

In [ ]:
import json

def toml_string(value):
    return json.dumps(str(value))

config_text = f'''[phase1]
dataset_path = {toml_string(DATASET_PATH)}
feature_table_path = {toml_string(FEATURE_TABLE_PATH)}
pause_durations_path = {toml_string(PAUSE_DURATIONS_PATH)}
output_dir = {toml_string(OUTPUT_DIR)}
seed = 20260822
validation_fraction = 0.1
noise_sigma = 0.05
phoneme_envelope = [0.3333333333, 0.6666666667, 1.0, 0.6666666667, 0.3333333333]
hidden_size = 128
num_layers = 1
dropout = 0.0
learning_rate = 0.001
gradient_clip_norm = 1.0
target_checkpoint_count = 30
'''
COLAB_CONFIG = pathlib.Path("/content/phase1_colab.toml")
COLAB_CONFIG.write_text(config_text, encoding="utf-8")
print(config_text)

## 5. Train
The command performs one developmental pass and writes periodic checkpoints plus validation metrics to local runtime storage in `OUTPUT_DIR`. Do not disconnect the runtime before the next cell downloads the archive.

In [ ]:
subprocess.run([sys.executable, "-m", "devlm.cli", "--config", str(COLAB_CONFIG)], check=True)

## 6. ZIP and download all training outputs
This archive includes every checkpoint plus metrics, the session split, vocabulary, and saved IPA feature mapping required for reproducibility.

In [ ]:
import shutil

checkpoint_files = sorted(pathlib.Path(OUTPUT_DIR).glob("checkpoint_step_*.pt"))
if not checkpoint_files:
    raise FileNotFoundError("No checkpoints found; complete the training cell first")
archive_path = shutil.make_archive("/content/devlm_phase1_outputs", "zip", root_dir=OUTPUT_DIR)
print(f"Archived {len(checkpoint_files)} checkpoints to {archive_path}")
files.download(archive_path)